# Nicheformer Pipeline: Tokenization → Embeddings → Downstream Tasks

This notebook demonstrates the complete pipeline **after preprocessing** using the Nicheformer project.

## Pipeline Overview

1. **Tokenization** — Load ready-to-tokenize H5AD files, tokenize gene expression via `sf_normalize → /technology_mean → _sub_tokenize_data`, save as Parquet files
2. **Embedding Extraction** — Load pretrained Nicheformer model, use `NicheformerDataset` to load data, extract cell embeddings via `model.get_embeddings()`
3. **Downstream Fine-Tuning** — Fine-tune the model for niche composition regression/classification using `FineTuningModel` + `MerlinDataModuleDistributed` (reads tokenized Parquet files from Part 1)

> **Prerequisite**: Run `preprocess_spatial.py` first (via `run_preprocess_spatial.sh`) to generate the ready-to-tokenize H5AD files.

---
## Part 1: Tokenization

After preprocessing, the data is saved as split-specific H5AD files (e.g., `{dataset_title}_train_ready_to_tokenize.h5ad`).

The tokenization pipeline:
1. Load the H5AD file for a given split (train/test)
2. Process in chunks (~10K cells each) to avoid OOM
3. For each chunk: `nan_to_num → sf_normalize (10K) → /technology_mean → _sub_tokenize_data (sort, keep top-4096, +30 aux offset)`
4. Attach metadata columns (assay, specie, modality, idx, author_cell_type, niche, region, X_niche_*)
5. Save as Parquet files with the schema expected by `MerlinDataModuleDistributed`

In [2]:
# ============================================================
# PART 1: Tokenization
# ============================================================
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import math
import numba
from scipy.sparse import issparse
from sklearn.utils import sparsefuncs

import pyarrow.parquet as pq
import pyarrow
from os.path import join
from tqdm import tqdm
import os

# ---------- Configuration ----------
DATASET_TITLE = "nanostring_cosmx_human_liver_Cancerous"  # change to your dataset
READY_DIR = "/mnt/172/wh/25-12/spatial/preprocessed/to_tokenize/CancerousLiver"  # output from preprocess_spatial.py step 15
TOKENIZED_DIR = "/mnt/172/wh/25-12/spatial/preprocessed/tokenized_output/CancerousLiver"
TECHNOLOGY_MEAN_PATH = "/root/code/25-12/nicheformer/data/model_means/cosmx_mean_script.npy"  # e.g., cosmx_mean_script.npy

os.makedirs(TOKENIZED_DIR, exist_ok=True)
os.makedirs(join(TOKENIZED_DIR, 'train'), exist_ok=True)
os.makedirs(join(TOKENIZED_DIR, 'test'), exist_ok=True)

In [ ]:


# ---------- Tokenization Functions (from dataset.py) ----------
def sf_normalize(X):
    X = X.copy()
    counts = np.array(X.sum(axis=1))
    counts += counts == 0.
    scaling_factor = 10000. / counts
    if issparse(X):
        sparsefuncs.inplace_row_scale(X, scaling_factor)
    else:
        np.multiply(X, scaling_factor.reshape((-1, 1)), out=X)
    return X

@numba.jit(nopython=True, nogil=True)
def _sub_tokenize_data(x, max_seq_len=4096, aux_tokens=30):
    scores_final = np.empty((x.shape[0], max_seq_len))
    for i, cell in enumerate(x):
        nonzero_mask = np.nonzero(cell)[0]
        sorted_indices = nonzero_mask[np.argsort(-cell[nonzero_mask])][:max_seq_len]
        sorted_indices = sorted_indices + aux_tokens
        scores = np.zeros(max_seq_len, dtype=np.int32)
        scores[:len(sorted_indices)] = sorted_indices.astype(np.int32)
        scores_final[i, :] = scores
    return scores_final

def tokenize_data(x, technology_mean, max_seq_len=4096):
    x = np.nan_to_num(x)
    x = sf_normalize(x)
    tech_mean = technology_mean.copy()
    tech_mean += tech_mean == 0
    x = x / tech_mean.reshape((1, -1))
    tokens = _sub_tokenize_data(x, max_seq_len, 30)
    return tokens.astype('i4')

# ---------- Load Technology Mean ----------
technology_mean = np.load(TECHNOLOGY_MEAN_PATH)
print(f"Technology mean shape: {technology_mean.shape}")

# ---------- Tokenize Each Split ----------
for SPLIT in ['train', 'test']:
    print(f"\n{'='*60}")
    print(f"Tokenizing {SPLIT} split")
    print(f"{'='*60}")

    h5ad_path = join(READY_DIR, f"{DATASET_TITLE}_{SPLIT}_ready_to_tokenize.h5ad")
    if not os.path.exists(h5ad_path):
        print(f"  WARNING: {h5ad_path} not found, skipping.")
        continue

    adata_split = ad.read_h5ad(h5ad_path)
    print(f"  Loaded: {adata_split.shape[0]} cells, {adata_split.shape[1]} genes")

    obs_split = adata_split.obs.reset_index().rename(columns={'index': 'idx'})
    obs_split['idx'] = obs_split['idx'].astype('i8')

    N_BATCHES = math.ceil(obs_split.shape[0] / 10_000)
    batch_indices = np.array_split(obs_split.index, N_BATCHES)
    chunk_len = len(batch_indices[0])
    print(f"  N_BATCHES: {N_BATCHES}, chunk_len: {chunk_len}")

    print(f"  Tokenizing...")
    for batch in tqdm(range(N_BATCHES)):
        start = batch * chunk_len
        end = chunk_len * (batch + 1)

        obs_tokens = obs_split.iloc[start:end].copy()

        X_batch = adata_split.X[start:end]
        if issparse(X_batch):
            X_batch = X_batch.toarray()
        tokenized = tokenize_data(X_batch, technology_mean, max_seq_len=4096)

        cols = ['assay', 'specie', 'modality', 'idx',
                'author_cell_type', 'niche', 'region']
        obs_tokens = obs_tokens[cols]

        obs_tokens['X'] = [tokenized[i, :] for i in range(tokenized.shape[0])]

        for i in np.arange(5):
            niche_key = f"X_niche_{i}"
            if niche_key in adata_split.obsm:
                niche = adata_split.obsm[niche_key][start:end]
                if issparse(niche):
                    niche = niche.toarray()
                obs_tokens[niche_key] = [niche[j, :] for j in range(niche.shape[0])]

        obs_tokens = obs_tokens.sample(frac=1)

        table = pyarrow.Table.from_pandas(obs_tokens)
        out_path = join(TOKENIZED_DIR, SPLIT, f'tokens-{batch}.parquet')
        pq.write_table(table, out_path, row_group_size=1024)

    print(f"  Done! {N_BATCHES} parquet files written to {join(TOKENIZED_DIR, SPLIT)}")

# ---------- Verify ----------
print("\n" + "="*60)
print("Verification: reading back a parquet file")
print("="*60)
df_check = pd.read_parquet(join(TOKENIZED_DIR, 'train', 'tokens-0.parquet'))
print(f"Shape: {df_check.shape}")
print(f"Columns: {list(df_check.columns)}")
print(f"X dtype: {df_check['X'].iloc[0].dtype}")
print(f"X length: {len(df_check['X'].iloc[0])}")
print(f"Sample:\n{df_check.head(2)}")

Technology mean shape: (20310,)

Tokenizing train split


---
## Part 2: Embedding Extraction

After tokenization, we have Parquet files ready for the Nicheformer model pipeline.

The embedding extraction pipeline:
1. Load the pretrained Nicheformer checkpoint
2. Use `NicheformerDataset` to load the ready-to-tokenize H5AD (handles tokenization internally)
3. Pass batches through the model and extract embeddings via `model.get_embeddings(batch, layer=-1)`
4. Store embeddings in an AnnData object or save as numpy arrays

In [ ]:
# ============================================================
# PART 2: Embedding Extraction
# ============================================================
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader

from nicheformer.models import Nicheformer
from nicheformer.data import NicheformerDataset

# ---------- Configuration ----------
config_emb = {
    'checkpoint_path': '/root/code/25-12/nicheformer/ckpt/nicheformer.ckpt',
    'data_path': join(READY_DIR, f"{DATASET_TITLE}_train_ready_to_tokenize.h5ad"),
    'technology_mean_path': TECHNOLOGY_MEAN_PATH,
    'output_path': '/mnt/172/wh/25-12/spatial/adata_with_embeddings.h5ad',
    'output_dir': '/mnt/172/wh/25-12/spatial/output/',
    'batch_size': 32,
    'max_seq_len': 1500,
    'aux_tokens': 30,
    'chunk_size': 1000,
    'num_workers': 4,
    'precision': 32,
    'embedding_layer': -1,
    'embedding_name': 'nicheformer_embeddings',
}

# ---------- Load Data ----------
pl.seed_everything(42)

adata = ad.read_h5ad(config_emb['data_path'])
technology_mean = np.load(config_emb['technology_mean_path'])

# Ensure required obs columns exist for context tokens
if 'modality' not in adata.obs.columns:
    adata.obs['modality'] = 4  # spatial
if 'specie' not in adata.obs.columns:
    adata.obs['specie'] = 5    # human
if 'assay' not in adata.obs.columns:
    adata.obs['assay'] = 8     # cosmx

print(f"Data: {adata.shape[0]} cells, {adata.shape[1]} genes")

# ---------- Create Dataset & DataLoader ----------
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=config_emb['max_seq_len'],
    aux_tokens=config_emb['aux_tokens'],
    chunk_size=config_emb['chunk_size'],
    metadata_fields={'obs': ['modality', 'specie', 'assay']}
)

dataloader = DataLoader(
    dataset,
    batch_size=config_emb['batch_size'],
    shuffle=False,
    num_workers=config_emb['num_workers'],
    pin_memory=True
)

print(f"Dataset: {len(dataset)} cells, {len(dataloader)} batches")

# ---------- Load Model ----------
model = Nicheformer.load_from_checkpoint(
    checkpoint_path=config_emb['checkpoint_path'],
    strict=False,
    weights_only=False
)
model.eval()
print(f"Model loaded: {type(model).__name__}")
print(f"  dim_model={model.hparams.dim_model}, nlayers={model.hparams.nlayers}")

trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    default_root_dir=config_emb['output_dir'],
    precision=config_emb['precision'],
)

# ---------- Extract Embeddings ----------
print("\nExtracting embeddings...")
embeddings = []
device = next(model.parameters()).device

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Embedding"):
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}
        emb = model.get_embeddings(
            batch=batch,
            layer=config_emb['embedding_layer'],
            with_context=False
        )
        embeddings.append(emb.cpu().numpy())

embeddings = np.concatenate(embeddings, axis=0)
print(f"Embedding shape: {embeddings.shape}")
print(f"  ({embeddings.shape[0]} cells x {embeddings.shape[1]} dims)")

# ---------- Save Results ----------
embedding_key = f"X_{config_emb['embedding_name']}"
adata.obsm[embedding_key] = embeddings
adata.write_h5ad(config_emb['output_path'])
print(f"\nEmbeddings saved to {config_emb['output_path']}")
print(f"  Key in adata.obsm: '{embedding_key}'")

Seed set to 42
/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Data: 368226 cells, 20310 genes


100%|██████████| 369/369 [07:33<00:00,  1.23s/it]


Dataset: 368226 cells, 11508 batches


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Model loaded: Nicheformer
  dim_model=512, nlayers=12

Extracting embeddings...


Embedding: 100%|██████████| 11508/11508 [1:17:00<00:00,  2.49it/s]


Embedding shape: (368226, 512)
  (368226 cells x 512 dims)

Embeddings saved to /mnt/172/wh/25-12/spatial/adata_with_embeddings.h5ad
  Key in adata.obsm: 'X_nicheformer_embeddings'


### Alternative: Embedding Extraction via MerlinDataModuleDistributed

If you have NVIDIA Merlin installed and want to use the distributed Parquet pipeline (as used in the project's `_embeddings.py`):

In [9]:
# ============================================================
# PART 2b: Embedding Extraction via MerlinDataModuleDistributed
# ============================================================
# Requires: nvtabular (NVIDIA Merlin)

try:
    from nicheformer.data.datamodules import MerlinDataModuleDistributed
    from nicheformer._embeddings import get_embeddings_model
    MERLIN_AVAILABLE = True
except ImportError:
    MERLIN_AVAILABLE = False
    print("Merlin not available. Install with: pip install nvtabular")

if MERLIN_AVAILABLE:
    config_emb_merlin = {
        'checkpoint_path': '/root/code/25-12/nicheformer/ckpt/nicheformer.ckpt',
        'fine_tuned_checkpoint_path': '/root/code/25-12/nicheformer/ckpt/nicheformer_ft.ckpt',
        'organ': 'everything',
        'batch_size': 32,
        'num_workers': 4,
        'precision': 32,
        'parquet_path': TOKENIZED_DIR,
        'output_dir': '/mnt/172/wh/25-12/spatial/embeddings_output',
    }
    # embeddings = get_embeddings_model(config_emb_merlin)
    print("Merlin pipeline ready. Uncomment to run.")
else:
    print("Using the simple NicheformerDataset approach from Part 2 above.")

Merlin pipeline ready. Uncomment to run.


---
## Part 3: Downstream Fine-Tuning

After tokenization, we can fine-tune the Nicheformer model for downstream tasks.

### Supported Tasks

| Task | `supervised_task` | Description |
|------|-------------------|-------------|
| Niche Regression | `niche_regression` | Predict niche composition vector (e.g., X_niche_0..4) |
| Niche Classification | `niche_classification` | Predict discrete niche type |
| Density Regression | `density_regression` | Predict cell density (scalar) |
| Binary Classification | `niche_binary_classification` | Binary per-cell-type prediction |
| Multi-class Classification | `niche_multiclass_classification` | Multi-class per-cell-type prediction |

In [9]:
# ============================================================
# PART 3: Downstream Fine-Tuning (via MerlinDataModuleDistributed)
# ============================================================
# This approach reads the tokenized Parquet files (from Part 1)
# using MerlinDataModuleDistributed, which handles distributed
# data loading and provides train/val/test dataloaders.
#
# Prerequisite: Part 1 must have been run to generate Parquet files
# in TOKENIZED_DIR/train/ and TOKENIZED_DIR/test/.

import os

# ---------------------------------------------------------------------------
# FIX 1: Suppress tensorboard/protobuf 'MessageFactory' error
# ---------------------------------------------------------------------------
# TensorBoard 2.17.1 + protobuf 4.25.x has a known incompatibility where
# tensorboard/compat/__init__.py tries to import 'notf' and fails with:
#   'MessageFactory' object has no attribute 'GetPrototype'
# We monkey-patch tensorboard.compat to provide a dummy 'notf' module.
import tensorboard.compat
if not hasattr(tensorboard.compat, 'notf'):
    class _NotF:
        pass
    tensorboard.compat.notf = _NotF()

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
import torch
import torch.distributed as dist
import tempfile
import numpy as np

from nicheformer.models._nicheformer import Nicheformer
from nicheformer.models._fine_tune_model import FineTuningModel
from nicheformer.data.datamodules import MerlinDataModuleDistributed

# ---------- Configuration ----------
config_ft = {
    # Path to tokenized Parquet files (output of Part 1)
    'tokenized_dir': TOKENIZED_DIR,
    'checkpoint_path': '/root/code/25-12/nicheformer/ckpt/nicheformer.ckpt',
    'output_dir': '/mnt/172/wh/25-12/spatial/checkpoints',
    'batch_size': 32,
    'num_workers': 4,
    'precision': 32,
    'max_epochs': 100,
    'lr': 1e-4,
    'warmup': 10,
    'gradient_clip_val': 1.0,
    'accumulate_grad_batches': 10,
    'supervised_task': 'niche_regression',
    'label': 'X_niche_1',
    'dim_prediction': 17,
    'n_classes': 10,
    'freeze': True,
    'extract_layers': [11],
    'function_layers': 'mean',
    'extractor': False,
    'reinit_layers': False,
    'regress_distribution': True,
    'pool': 'mean',
    'predict_density': False,
    'ignore_zeros': False,
    'organ': 'liver',
}

# ---------- Load Pretrained Model ----------
pl.seed_everything(42)

backbone = Nicheformer.load_from_checkpoint(
    checkpoint_path=config_ft['checkpoint_path'],
    strict=False,
    weights_only=False,
)
print(f"Backbone loaded: {type(backbone).__name__}")

# ---------- Create FineTuningModel ----------
fine_tune_model = FineTuningModel(
    backbone=backbone,
    supervised_task=config_ft['supervised_task'],
    label=config_ft['label'],
    dim_prediction=config_ft['dim_prediction'],
    n_classes=config_ft['n_classes'],
    freeze=config_ft['freeze'],
    extract_layers=config_ft['extract_layers'],
    function_layers=config_ft['function_layers'],
    extractor=config_ft['extractor'],
    reinit_layers=config_ft['reinit_layers'],
    regress_distribution=config_ft['regress_distribution'],
    pool=config_ft['pool'],
    predict_density=config_ft['predict_density'],
    ignore_zeros=config_ft['ignore_zeros'],
    organ=config_ft['organ'],
    lr=config_ft['lr'],
    warmup=config_ft['warmup'],
    max_epochs=config_ft['max_epochs'],
    baseline=False,
    without_context=True,
)
print(f"FineTuningModel created: task={config_ft['supervised_task']}")
print(f"  freeze_backbone={config_ft['freeze']}")
if config_ft['freeze']:
    print(f"  Trainable params: {sum(p.numel() for p in fine_tune_model.linear_head.parameters())}")
else:
    print(f"  Total params: {sum(p.numel() for p in fine_tune_model.parameters())}")

# ---------- Configure MerlinDataModuleDistributed ----------
# The columns must match PARQUET_SCHEMA in datamodules.py.
# MerlinDataModuleDistributed reads Parquet files from:
#   {tokenized_dir}/train/  -> train dataloader
#   {tokenized_dir}/test/   -> val & test dataloaders (when splits=True)
#
# FineTuningModel.on_after_batch_transfer() will:
#   1. Add modality/assay/specie context tokens
#   2. Extract the label column (config_ft['label'])
#   3. Add CLS token

label_key = [
    'assay', 'specie', 'modality', 'niche', 'idx', 'X',
    'X_niche_0', 'X_niche_1', 'X_niche_2', 'X_niche_3', 'X_niche_4'
]

# ---------------------------------------------------------------------------
# FIX 2: Initialize distributed process group for single-GPU mode
# ---------------------------------------------------------------------------
# MerlinDataModuleDistributed.val_dataloader() calls get_rank(), which
# requires a default process group. For single-process mode, we use 'gloo'
# backend with a temp-file-based init method to avoid port conflicts
# (unlike tcp://localhost:23456 which can fail if port is in use).
if not dist.is_initialized():
    tmp_file = os.path.join(tempfile.gettempdir(), f"nicheformer_ft_{os.getpid()}.store")
    dist.init_process_group(
        backend='gloo',
        init_method=f'file://{tmp_file}',
        rank=0,
        world_size=1,
    )
    print(f"Distributed process group initialized (gloo, file://{tmp_file})")

# ---------- Create MerlinDataModuleDistributed ----------
dm = MerlinDataModuleDistributed(
    path=config_ft['tokenized_dir'],
    batch_size=config_ft['batch_size'],
    world_size=config_ft['num_workers'],
    columns=label_key,
    splits=True,
    # supervised_task=config_ft['supervised_task'],
)
dm.setup(stage='fit')
print(f"Train samples: {len(dm.train_dataloader())} batches")
print(f"Val samples: {len(dm.val_dataloader())} batches")

# ---------- Configure Trainer ----------
# TensorBoard logger is safe now because FIX 1 monkey-patches the protobuf issue
tb_logger = TensorBoardLogger(
    save_dir=config_ft['output_dir'],
    name='nicheformer_ft_logs',
)
trainer = pl.Trainer(
    max_epochs=config_ft['max_epochs'],
    accelerator='auto',
    devices=1,
    precision=config_ft['precision'],
    gradient_clip_val=config_ft['gradient_clip_val'],
    accumulate_grad_batches=config_ft['accumulate_grad_batches'],
    logger=tb_logger,
    callbacks=[
        ModelCheckpoint(
            dirpath=config_ft['output_dir'],
            filename='nicheformer-ft-{epoch:02d}-{val_loss:.4f}',
            monitor='val_loss',
            mode='min',
            save_top_k=3,
        )
    ],
)
print(f"Trainer configured with TensorBoard logger")
print(f"  Logs: {tb_logger.log_dir}")
print(f"  Run: tensorboard --logdir={config_ft['output_dir']}/nicheformer_ft_logs")

# ---------- Train ----------
try:
    trainer.fit(fine_tune_model, dm)
finally:
    # Cleanup: remove temp file used for distributed init
    if dist.is_initialized():
        dist.destroy_process_group()
        print("Distributed process group destroyed.")
    tmp_file = os.path.join(tempfile.gettempdir(), f"nicheformer_ft_{os.getpid()}.store")
    if os.path.exists(tmp_file):
        os.remove(tmp_file)
        print(f"Cleaned up temp file: {tmp_file}")

print("Fine-tuning complete!")


Seed set to 42


Backbone loaded: Nicheformer
Backbone
Linear(in_features=512, out_features=17, bias=False)
FineTuningModel created: task=niche_regression
  freeze_backbone=True
  Trainable params: 8704
Distributed process group initialized (gloo, file:///tmp/nicheformer_ft_26006.store)


/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/merlin/io/dataset.py:267: UserWarning: Initializing an NVTabular Dataset in CPU mode.This is an experimental feature with extremely limited support!
  warnings.warn(
/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/merlin/io/dataset.py:267: UserWarning: Initializing an NVTabular Dataset in CPU mode.This is an experimental feature with extremely limited support!
  warnings.warn(
/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/merlin/io/dataset.py:267: UserWarning: Initializing an NVTabular Dataset in CPU mode.This is an experimental feature with extremely limited support!
  warnings.warn(
/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/merlin/io/dataset.py:267: UserWarning: Initializing an NVTabular Dataset in CPU mode.This is an experimental feature with extremely limited support!
  warnings.warn(
/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/merlin/io/datase

Train samples: 3110 batches
Val samples: 865 batches
Trainer configured with TensorBoard logger
  Logs: /mnt/172/wh/25-12/spatial/checkpoints/nicheformer_ft_logs/version_0
  Run: tensorboard --logdir=/mnt/172/wh/25-12/spatial/checkpoints/nicheformer_ft_logs


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type        ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone    │ Nicheformer │ 49.3 M │ eval  │     0 │
│ 1 │ linear_head │ Linear      │  8.7 K │ train │     0 │
│ 2 │ softmax     │ Softmax     │      0 │ train │     0 │
│ 3 │ cls_loss    │ MSELoss     │      0 │ train │     0 │
└───┴─────────────┴─────────────┴────────┴───────┴───────┘

...

/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:534: Found 142 
module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is 
intentional, you can ignore this warning.


Detected KeyboardInterrupt, attempting graceful shutdown ...


Distributed process group destroyed.


SystemExit: 1

/root/code/25-12/nicheformer/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


---
## Appendix: Key Architecture Details

### Token IDs

| Token | ID Range | Description |
|-------|----------|-------------|
| Padding/Mask | 0 | Padding token |
| CLS | 2 | Classification token (optional) |
| Auxiliary | 0-29 | Reserved for special tokens |
| Genes | 30-20369 | Gene tokens (after +30 aux offset) |

### Context Token IDs (prepended to sequence)

| Token | ID | Description |
|-------|----|-------------|
| Modality: spatial | 4 | Spatial transcriptomics |
| Modality: dissociated | 3 | scRNA-seq |
| Specie: human | 5 | Homo sapiens |
| Specie: mouse | 6 | Mus musculus |
| Assay: cosmx | 8 | NanoString CosMx |
| Assay: merfish | 7 | MERFISH |
| Assay: xenium | 9 | 10x Xenium |

### Model Architecture

| Parameter | Value |
|-----------|-------|
| Transformer layers | 12 |
| Attention heads | 16 |
| Embedding dimension | 512 |
| Feed-forward dimension | 1024 |
| Vocabulary size | 20340 |
| Max context length | 1500 |
| Dropout | 0.0 |
| Masking probability (MLM) | 0.15 |

### Data Flow Summary

\`\`\`
Raw H5AD
  │
  ▼
preprocess_spatial.py (15 steps)
  │  • Gene mapping (symbol → Ensembl)
  │  • CELLxGENE schema validation
  │  • Niche composition (X_niche_0..4)
  │  • Gene reordering (via model.h5ad concat)
  │  • Technology mean loading
  │  • Obs → token ID mapping
  │  • Split & save ready-to-tokenize H5AD
  │
  ▼
Tokenization (Part 1)
  │  • sf_normalize → /tech_mean → _sub_tokenize
  │  • Save as Parquet (train/ & test/ dirs)
  │
  ├──► Embedding Extraction (Part 2)
  │     • NicheformerDataset + Nicheformer model
  │     • model.get_embeddings() → adata.obsm
  │
  └──► Fine-Tuning (Part 3)
        • MerlinDataModuleDistributed (reads Parquet from Part 1)
        • FineTuningModel wraps Nicheformer backbone
        • trainer.fit(model=..., datamodule=module)
        • Predict niche composition / class
\`\`\`